In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [76]:
df = pd.read_csv('kc_house_data-checkpoint.csv')

In [77]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [80]:
X = df.drop(columns=['id', 'date', 'price'])
y = df['price']

In [81]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [82]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


lr = LinearRegression()
lr.fit(X_train, y_train)


y_pred_lr = lr.predict(X_test)


r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("R2:", r2_lr)
print("MSE:", mse_lr)
print("RMSE:", rmse_lr)


R2: 0.7011904448878294
MSE: 45173046132.79195
RMSE: 212539.51663818178


In [83]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [84]:
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)
    

In [85]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_log)


LinearRegression()

In [86]:
y_pred_log = lr.predict(X_test_scaled)
y_pred = np.exp(y_pred_log)
    

In [87]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_log)


Ridge()

In [88]:
for a in [0.1, 1, 10, 50]:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_scaled, y_train_log)
    preds = np.exp(ridge.predict(X_test_scaled))
    print(a, r2_score(y_test, preds))


0.1 0.4986509000483741
1 0.49864405809323153
10 0.49857641012810394
50 0.4982924885484745


In [89]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)


In [95]:
from sklearn.metrics import r2_score, mean_squared_error

ridge = Ridge(alpha=0.1)
ridge.fit(X_train_poly, y_train_log)

y_pred = np.exp(ridge.predict(X_test_poly))

r2_lr = r2_score(y_test, y_pred)
mse_lr = mean_squared_error(y_test, y_pred)
rmse_lr = np.sqrt(mse_lr)

print("R2:", r2_lr)
print("MSE:", mse_lr)
print("RMSE:", rmse_lr)

R2: 0.803666899581186
MSE: 29680992628.51157
RMSE: 172281.72459234198


In [96]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

In [97]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [98]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

KNeighborsRegressor()

In [99]:
y_pred_knn = knn.predict(X_test_scaled)

r2_knn = r2_score(y_test, y_pred_knn)
mse_knn = mean_squared_error(y_test, y_pred_knn)
rmse_knn = np.sqrt(mse_knn)

print("\nKNN Regression Results")
print("R2:", r2_knn)
print("MSE:", mse_knn)
print("RMSE:", rmse_knn)


KNN Regression Results
R2: 0.7798089509757427
MSE: 33287758859.875557
RMSE: 182449.3323086592


In [100]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "KNN Regression"],
    "R2 Score": [r2_lr, r2_knn],
    "RMSE": [rmse_lr, rmse_knn]
})

comparison


,Model,R2 Score,RMSE
0,Linear Regression,0.803667,172281.724592
1,KNN Regression,0.779809,182449.332309


In [101]:
df['price_category'] = pd.qcut(
    df['price'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

df['price_category'].value_counts()

price_category
Low       7226
Medium    7223
High      7164
Name: count, dtype: int64

In [102]:
X = df.drop(columns=['id', 'date', 'price', 'price_category'])
y = df['price_category']

In [103]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [104]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [105]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train_scaled, y_train)


KNeighborsClassifier()

In [106]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = knn_clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7807078417765441

Classification Report:
               precision    recall  f1-score   support

        High       0.81      0.81      0.81      1433
         Low       0.85      0.85      0.85      1445
      Medium       0.69      0.69      0.69      1445

    accuracy                           0.78      4323
   macro avg       0.78      0.78      0.78      4323
weighted avg       0.78      0.78      0.78      4323



In [107]:
confusion_matrix(y_test, y_pred)

array([[1160,   17,  256],
       [  27, 1223,  195],
       [ 252,  201,  992]], dtype=int64)

In [108]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("KNN Classification Accuracy:", accuracy)

KNN Classification Accuracy: 0.7807078417765441


In [109]:
df.head(4)

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price_category
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,1180,0,1955,0,98178,47.5112,-122.257,1340,5650,Low
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639,Medium
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,770,0,1933,0,98028,47.7379,-122.233,2720,8062,Low
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,1050,910,1965,0,98136,47.5208,-122.393,1360,5000,High
